# Data Mining Versuch Recommender Systeme

* Autor: Prof. Dr. Johannes Maucher
* Datum: 30.09.2015

## Abgabe:

- **Abzugeben ist das Jupyter Notebook mit dem verlangten Implementierungen und den entsprechenden Ausgaben.**
- **Das Notebook ist als .ipynb und als .html abzugeben.**
- **Klausurelevante Fragen sind Dokument "Fragenkatalog Datamining" zu finden.**
- Antworten auf Fragen im Notebook, Diskussionen und Beschreibung der Ergebnisse sind optional (aber empfohlen) und werden nicht bewertet.

* [Übersicht Data Mining Praktikum](https://maucher.pages.mi.hdm-stuttgart.de/ai/page/dm/)


# Einführung

## Lernziele:

In diesem Versuch sollen Kenntnisse in folgenden Themen vermittelt werden:

* __Ähnlichkeit:__ Verfahren zur Bestimmung der Ähnlichkeit zwischen Personen (Kunden) und Elementen (Produkten)
* __Empfehlungssysteme__ Collaborative Filtering 
* __Collaborative Filtering:__ Nutzerbezogener Ansatz und elementbasierter Ansatz

Sämtliche Verfahren und Algorithmen werden in Python implementiert.

## Theorie zur Vorbereitung

### Recommender Systeme

Recommender Systeme werden im E-Commerce eingesetzt um Werbung in Form von kundenspezifischen Empfehlungen zu verteilen. Weitläufig bekannt sind die Amazon-Empfehlungen, die entweder per e-mail geschickt oder nach dem Log-In in der Web-Page angezeigt werden. Diese Empfehlungen werden in Abhängigkeit von den bisher vom jeweiligen Kunden gekauften bzw. bewerteten Produkten erstellt. In diesem Versuch werden die derzeit wohl am weitest verbreiteteten Verfahren für die Erzeugung kundenspezifischer Empfehlungen vorgestellt, darunter das elementweise Collaborative Filtering, welches z.B. auch von Amazon eingesetzt wird.     

Direkt-Marketing Methoden wie die kundenspezifische Erzeugung und Bereitstellung von Werbung erfordern detaillierte Kunden- und Warenkorbanalysen. Kunden mit ähnlichem Kaufverhalten werden in Kundengruppen zusammengefasst. Die Warenkorbanalyse untersucht u.a. welche Waren bevorzugt im Verbund von der gleichen Person gekauft werden. Damit kann ein Händler Werbung in Form von Empfehlungen individuell und gezielt an seine Kunden richten, abhängig davon welcher Kundengruppe er angehört und welche Produkte bevorzugt von dieser Kundengruppe nachgefragt werden. 

Im ersten Teil der Übung werden fiktive Daten in einer überschaubaren Menge verwendet. Es handelt sich hier um Filmbewertungen. Anhand dieses Beispiels sollen die notwendigen Methoden und Abläufe implementiert und getestet werden. Diese werden im zweiten Teil der Übung auf echte Daten angewandt. Hierzu werden über eine Python-API Daten vom Internet-Meta-Radio _last.fm_ integriert. Auf der Basis dieser Daten sollen dann Musikempfehlungen für last.fm User berechnet werden. 

Recommender Systeme lassen sich mit

* Clustering Verfahren
* Suchalgorithmen
* Collaborativen Filtering 
 
realisieren. Am häufigsten wird hierbei das Collaborative Filtering eingesetzt. Für das Collaborative Filtering wird jeder der $M$ User durch einen $N$-dimensionalen Vektor beschrieben, wobei $N$ die Anzahl der Produkte im Angebot des Händlers ist. Jedes Element im Vektor gehört zu einem speziellen Produkt. Das Element hat den Wert 1, wenn der User dieses Produkt bereits gekauft hat, sonst 0 (andere Wertbelegungen sind möglich, z.B. wenn Produktbewertungen vorliegen). Alle $M$ Zeilenvektoren können zur _User/Item_ Matrix zusammengefasst werden (siehe Abbildung).


<img src="https://maucher.home.hdm-stuttgart.de/Pics/UserItemMatrix.png" style="width:500px" align="center">

Das traditionelle **userbasierte Collaborative Filtering (UCF)**, benutzt die Ähnlichkeit zwischen Benutzern: Um für User $U_i$ eine Empfehlung zu erzeugen wird zunächst der diesem User ähnlichste Kunde (oder eine Menge vom ähnlichsten Kunden) ermittelt. Dann werden $U_i$ die Produkte (Items) empfohlen, welche der ähnlichste Kunde gekauft hat, $U_i$ selbst jedoch noch nicht. 

Dieser Ansatz skaliert schlecht im Fall sehr großer *User/Item*-Matrizen. Ausserdem ist er für User, welche erst wenige Produkte gekauft haben unzuverlässig. Besser eignet sich in diesen Fällen das **itembasierte Collaborative Filtering (ICF)**. Es wird u.a. von Amazon.com eingesetzt. Diese Variante benutzt die Ähnlichkeit zwischen Produkten (Items). Dabei sind Produkte umso ähnlicher je mehr Kunden diese Produkte gemeinsam gekauft haben. Für die Produkte welche ein Referenzuser $U_i$ bereits gekauft hat, werden die ähnlichsten Produkte ermittelt. Diese ähnlichsten Produkte werden $U_i$ empfohlen, wenn er sie nicht schon selbst gekauft hat.

Im folgenden Abschnitt werden einige gebräuchliche Metriken für die Berechnung der Ähnlichkeit zwischen Benutzern oder Artikeln vorgestellt. Für Collaboratives Filtering wird sehr häufig das Cosinus - Ähnlichkeitsmaß eingesetzt.


### Gebräuchliche Ähnlichkeitsmaße

Die __euklidische Distanz__ $d_E(\underline{a},\underline{b})$ zwischen zwei n-dimensionalen Vektoren $\underline{a}=(a_1,\ldots,a_n)$ und $\underline{b}=(b_1,\ldots,b_n)$ berechnet sich zu
	$$
	d_E(\underline{a},\underline{b})=\sqrt{\sum_{i=1}^n (a_i-b_i)^2}
	$$
Zwei Vektoren können als umso ähnlicher erachtet werden, je kleiner deren euklidische Distanz ist. 
Ein auf der euklidischen Metrik basierendes Ähnlichkeitsmaß zwischen zwei Vektoren $\underline{a}$ und $\underline{b}$ kann durch 
$$
s_E(\underline{a},\underline{b})=\frac{1}{1+d_E(\underline{a},\underline{b})}
$$
angegeben werden.


__Pearson Korrelation__
Die Ähnlichkeit zwischen zwei Vektoren kann auch durch den Pearson-Korrelationskoeffizient $\rho_{\underline{a},\underline{b}}$ ausgedrückt werden. Er berechnet sich zu
$$
\rho_{\underline{a},\underline{b}}= \frac{1}{N}\cdot \sum\limits_{i=1}^{N}\frac{(a_i-\overline{a})}{\sigma_a} \frac{(b_i-\overline{b})}{\sigma_b}
$$
Dabei bezeichnet $N$ die Länge der Vektoren, $\overline{a}$ den Mittelwert und $\sigma_a$ die Standardabweichung des Vektors $\underline{a}$. 

Der Pearson-Korrelationskoeffizient misst die lineare Abhängigkeit zwischen zwei Vektoren. Der maximale Wert von $+1$ wird erreicht, wenn die durch die beiden Vektoren definierten N Punkte im 2-dimensionalen Raum auf einer ansteigenden Geraden liegen. Der Minimalwert von $-1$ wird erreicht, wenn die Punkte auf einer abfallenden Geraden liegen. Der Betrag des Koeffizienten ist umso kleiner, je stärker die Punkte von einer fiktiven Geraden (kann durch lineare Regression berechnet werden) abweichen. Der Koeffizient ist $0$ wenn keine lineare Abhängigkeit zwischen den Vektoren besteht.


__Cosinus Ähnlichkeitsmaß__
Die Ähnlichkeit zwischen zwei Vektoren kann auch durch den Cosinus $\cos(\underline{a},\underline{b})$ ausgedrückt werden. Er berechnet sich zu
$$
\cos(\underline{a},\underline{b})= \frac{\underline{a} \cdot \underline{b}}{\left\|\underline{a}\right\|\cdot \left\|\underline{b}\right\|}
$$
wobei im Zähler das Skalarprodukt der beiden Vektoren steht und mit $\left\|\underline{x}\right\|$ der Betrag des Vektors $\underline{x}$ bezeichnet wird.

Falls die Vektoren $\underline{a}$ und $\underline{b}$ mittelwertfrei sind, ist der Cosinus-Ähnlichkeitswert gleich dem Pearson-Korrelationswert. In der Dokument- und Textanalyse wird vornehmlich das Cosinus-Ähnlichkeitsmaß verwendet. 


__Russel Rao Ähnlichkeitsmaß__
Die Russel Rao-Ähnlichkeit zwischen zwei binären Vektoren $\underline{a}$ und $\underline{b}$ mißt das Verhältnis zwischen der Anzahl $\alpha$ der Stellen in denen beide Vektoren den Wert 1 haben und der Länge $n$ der Vektoren. Z.B. ist für die Vektoren $\underline{a}=(1,0,1,0,0,1)$ und $\underline{b}=(0,1,1,1,0,1)$ die Russel-Rao-Ähnlichkeit $s_{RR}(\underline{a},\underline{b})=2/6=0.333$.

__Jaccard Ähnlichkeitsmaß__
Die Jaccard-Ähnlichkeit zwischen zwei binären Vektoren $\underline{a}$ und $\underline{b}$ mißt das Verhältnis zwischen der Anzahl $\alpha$ der Stellen in denen beide Vektoren den Wert $1$ haben und der Anzahl der Stellen in denen mindestens einer der beiden Vektoren ungleich $0$ ist. Z.B. ist für die Vektoren $\underline{a}=(1,0,1,0,0,1)$ und $\underline{b}=(0,1,1,1,0,1)$ die Jaccard-Ähnlichkeit $s_{J}(\underline{a},\underline{b})=2/5=0.4$. Die Jaccard Metrik wird in diesem Versuch für die Bestimmung der Ähnlichkeit von *last.fm*-Usern eingesetzt.


## Vor dem Versuch zu klärende Fragen
Eine Untermenge der im Folgenden aufgeführten Fragen wird zu Beginn des Versuchs im Rahmen eines Gruppenkolloqs abgefragt. Auf jede Frage sollte von mindestens einem Gruppenmitglied eine Antwort geliefert werden und jedes Gruppenmitglied muss mindestens eine der gestellten Fragen beantworten können.

**Aufgaben:**

* Beschreiben Sie das Prinzip des userbasierten Collaborativen Filtering (UCF).

Das userbasierte collaborative Filtering (UCF) ist eine Technik, bei der User anhand ihrer Präferenzen gruppiert werden und anhand dieser Gruppen Empfehlungen vorgeschlagen bekommt.

Da man eine Anzahl von `n` Usern und `n` Elementen hat, wird eine Matrix verwendet, um so das Benutzerverhalten aller User darzustellen. Dabei hat jeder User eine Zeile dieser Matrix. Anschließend kann man mithilfe eines Distanzmaßes gemessen werden, wie ähnlich bestimmte User zueinander sind und diese User gruppieren. Anhand der Gruppierung kann den Usern die Produkte empfohlen werden, die sie bisher nicht gekauft haben [1].

* Welche Nachteile hat das UCF?

Der größte Nachteil beim UCF ist, dass sobald neue User erstellt werden, dieser User keine Interaktionen im System hat und somit nicht in die richtige Gruppe gruppiert werden kann. Somit würde ein neuer User keine passenden Vorschläge für sich selbst haben.

Zudem kann Datenknappheit auch ein Problem sein, da in großen System viele Elemente keine Daten zur Verfügung haben. Wenn große Teile der Matrix leer bleiben, kann es deutlich schwieriger werden, darunter feste Muster zu finden. Somit kann es bei der Empfehlung von Produkten zu Ungenauigkeiten kommen. [1]

* Worin besteht der Unterschied zwischen UCF und itembasierten Collaborativen Filtering (ICF)?

Hier geht es nicht mehr um die Ähnlichkeit der User, sondern um die Ähnlichkeit Produkte. Die Ähnlichkeit wird daran bestimmt, welche Produkte wie oft gemeinsam gekauft werden. Wenn also ein User ein Produkt kauft, werden ihm ähnliche Produkte empfohlen, die er noch nicht gekauft hat.

* Gegeben seien die Vektoren 

    $$
    \underline{a}  =  [1,2,3,4,5,6] \\
    \underline{b}  =  [3,3,5,6,7,8] \\
    $$
    
    Schreiben Sie eine Python Funktion, die den Mittelwert derartiger Vektoren berechnet. Schreiben Sie eine weitere Funktion, die die Varianz berechnet

In [59]:
v1 = [1,2,3,4,5,6]
v2 = [3,3,5,6,7,8]

In [60]:
# Mittelwert berechnen 
def mean(vector):
    return sum(vector) / len(vector)

In [61]:
# Varianz berechnen 
def variance(vector):
    sum = 0 
    m = mean(vector)
    for item in vector:
        sum = sum + (item - m)**2
    return sum / len(vector)

In [62]:
print("Vektor 1: ", mean(v1))
print("Vektor 2: ", mean(v2))

print("Varianz 1: ", variance(v1))
print("Varianz 2: ", variance(v2))

Vektor 1:  3.5
Vektor 2:  5.333333333333333
Varianz 1:  2.9166666666666665
Varianz 2:  3.5555555555555554


* Wie groß ist die

    - Euklidische Ähnlichkeit
    - Pearson Ähnlichkeit
    - Cosinus Ähnlichkeit
    
    zwischen den Vektoren $\underline{a}$ und $\underline{b}$? 

In [63]:

import math

# Euklidische Distanz
def euclidean_distance(vectorA, vectorB):
    sum = 0
    for i in range(len(vectorA)):
        sum = sum + (vectorA[i] - vectorB[i])**2
    return math.sqrt(sum)

In [64]:
# Euklidische Ähnlichkeit
def euclidean_similarity(vectorA, vectorB):
    sum = 1 / (1 + euclidean_distance(vectorA, vectorB))
    return round(sum, 3)

In [65]:
# Pearson Korrelation
def pearson_correlation(vectorA, vectorB):
    n = len(vectorA)
    meanA = mean(vectorA)
    meanB = mean(vectorB)
    standard_deviationA = math.sqrt(variance(vectorA))
    standard_deviationB = math.sqrt(variance(vectorB))

    sum = 0
    for i in range(n):
        sum = sum + ((vectorA[i] - meanA) / standard_deviationA) * ((vectorB[i] - meanB) / standard_deviationB)

    return round(sum / n, 3)

In [66]:
# Betrag eines Vektors 
def vector_norm(vector):
    sum = 0
    for item in vector:
        sum = sum + item**2
    return math.sqrt(sum)

In [67]:
# Sklarprodukt
def scalar_product(vectorA, vectorB):
    sum = 0
    for i in range(len(vectorA)):
        sum = sum + vectorA[i] * vectorB[i]
    return sum

In [68]:
# Cosinus Ähnlichkeit
def cosine_similarity(vectorA, vectorB):
    vectorA_norm = vector_norm(vectorA)
    vectorB_norm = vector_norm(vectorB)

    sum = scalar_product(vectorA, vectorB) / (vectorA_norm * vectorB_norm)
    return round(sum, 3)

In [69]:
print("Euklidische Ähnlichkeit: ", euclidean_similarity(v1, v2))
print("Pearson Korrelation: ", pearson_correlation(v1, v2))
print("Cosinus Ähnlichkeit: ", cosine_similarity(v1, v2))

Euklidische Ähnlichkeit:  0.179
Pearson Korrelation:  0.983
Cosinus Ähnlichkeit:  0.991


* In welchen Fällen sind Cosinus- und Pearsonähnlichkeit der euklidischen Ähnlichkeit vorzuziehen?

Die euklidische Distanz berechnet die exakte Distanz zwei verschiedener Punkte, während Cosinus- und Pearsonähnlichkeit die Richtungskorrelation untersucht. Die euklidische Distanz eignet sich also mehr für Vergleiche absoluter Werte (z.B. Distanzen in Luftlinien), während Cosinus- und Pearsonähnlichkeit Muster und Relationen untersucht, wie beispielsweise Nutzerbewertungen oder Textvektoren.

In [70]:
from IPython.display import Latex
from IPython.display import Image
import pylast

# Versuchsdurchführung
## Teil 1: Fiktive Filmbewertung
### Daten
Folgende Tabelle enthält die Filmbewertungen von 7 Personen.
from IPython.display import Latex
In diesem Versuch sollen Kenntnisse in folgenden Themen vermittelt werden:


<img src="https://maucher.home.hdm-stuttgart.de/Pics/recommenderFilmRecommendations.PNG" style="width:500px" align="center">



Die Tabelle ist als Python dictionary _critics_ implementiert. Die Keys des Python-Dictionary definieren die Namen von Personen (Zeilen in der Matrix), die Filme bewertet haben. Die Values sind selbst wieder Dictionarys, welche als Keys die Filmnamen (Spalten in der Matrix) und als Values die jeweilige Filmbewertung (Matrixelment) enthalten.

In [71]:
critics={'Lisa Rose': {'Lady in the Water': 2.5, 'Snakes on a Plane': 3.5,
 'Just My Luck': 3.0, 'Superman Returns': 3.5, 'You, Me and Dupree': 2.5, 
 'The Night Listener': 3.0},
'Gene Seymour': {'Lady in the Water': 3.0, 'Snakes on a Plane': 3.5, 
 'Just My Luck': 1.5, 'Superman Returns': 5.0, 'The Night Listener': 3.0, 
 'You, Me and Dupree': 3.5}, 
'Michael Phillips': {'Lady in the Water': 2.5, 'Snakes on a Plane': 3.0,
 'Superman Returns': 3.5, 'The Night Listener': 4.0},
'Claudia Puig': {'Snakes on a Plane': 3.5, 'Just My Luck': 3.0,
 'The Night Listener': 4.5, 'Superman Returns': 4.0, 
 'You, Me and Dupree': 2.5},
'Mick LaSalle': {'Lady in the Water': 3.0, 'Snakes on a Plane': 4.0, 
 'Just My Luck': 2.0, 'Superman Returns': 3.0, 'The Night Listener': 3.0,
 'You, Me and Dupree': 2.0}, 
'Jack Matthews': {'Lady in the Water': 3.0, 'Snakes on a Plane': 4.0,
 'The Night Listener': 3.0, 'Superman Returns': 5.0, 'You, Me and Dupree': 3.5},
'Toby': {'Snakes on a Plane':4.5,'You, Me and Dupree':1.0,'Superman Returns':4.0}
}

### Ähnlichkeiten berechnen

Für die Bestimmung der Ähnlichkeit zwischen Personen und Produkten werden in diesem Versuch ein auf der euklidischen Distanz basierendes Ähnlichkeitsmaß und die Pearson-Korrelation verwendet. Beide Ähnlichkeitsmaße sind in den unten definierten Funktionen implementiert. Alle drei hier implementierten Funktionen zur Berechnung der Ähnlichkeit erhalten als Übergabeparameter das oben definierte Dictionary, das die Filmbewertungen enthält und die Namen der zwei Personen, die verglichen werden sollen. 

Zu beachten ist, dass in beiden Funktionen für die Berechnung der Ähnlichkeit zwischen zwei Personen nur die Produkte berücksichtigt werden, welche von beiden Personen schon bewertet wurden. Es handelt sich hier also um modifizierte Ähnlichkeitsfunktionen. 

__Aufgabe:__
Fragen Sie von diesem Dictionary _Toby's_ Bewertung des Films _Snakes on a Plane_ ab und geben Sie diesen Wert aus: 

In [72]:
print("Tobys Bewertungen zum Film 'Snakes on a Plane': ", critics['Toby']['Snakes on a Plane'])

Tobys Bewertungen zum Film 'Snakes on a Plane':  4.5


In [73]:
import numpy as np
import scipy.spatial.distance as sci


def sim_euclid(prefs,person1,person2,normed=True):
    ''' Returns a euclidean-distance-based similarity score for 
    person1 and person2. In the distance calculation the sum is computed 
    only over those items, which are nonzero for both instances, i.e. only
    films which are ranked by both persons are regarded.
    If the parameter normed is True, then the euclidean distance is divided by
    the number of non-zero elements integrated in the distance calculation. Thus
    the effect of larger distances in the case of an increasing number of commonly ranked
    items is avoided.
    '''
    # Get the list of shared_items
    si={}
    for item in prefs[person1]: 
        if item in prefs[person2]: 
            si[item] = 1
            
    # len(si) counts the number of common ratings
    # if they have no ratings in common, return 0
    if len(si) == 0: 
        return 0

    # Add up the squares of all the differences
    sum_of_squares=np.sqrt(sum([pow(prefs[person1][item]-prefs[person2][item],2) 
                     for item in prefs[person1] if item in prefs[person2]]))
    if normed:
        sum_of_squares= 1.0/len(si)*sum_of_squares
    return 1/(1+sum_of_squares)


def sim_pearson(prefs,p1,p2):
    '''
    Returns the Pearson correlation coefficient for p1 and p2
    '''

    # Get the list of commonly rated items
    si={}
    for item in prefs[p1]: 
        if item in prefs[p2]: 
            si[item]=1

    # if they are no ratings in common, return 0
    if len(si)==0: return 0

    # Sum calculations
    n=len(si)

    # Calculate means of person 1 and 2
    mp1=np.mean([prefs[p1][it] for it in si])
    mp2=np.mean([prefs[p2][it] for it in si])

    # Calculate standard deviation of person 1 and 2
    sp1=np.std([prefs[p1][it] for it in si])
    sp2=np.std([prefs[p2][it] for it in si])

    # If all elements in one sample are identical, the standard deviation is 0. 
    # In this case there is no linear correlation between the samples
    if sp1==0 or sp2==0:
        return 0
    r=1/(n*sp1*sp2)*sum([(prefs[p1][it]-mp1)*(prefs[p2][it]-mp2) for it in si])
    return r


def sim_RusselRao(prefs,person1,person2,normed=True):
    ''' Returns RusselRao similaritiy between 2 users. The RusselRao similarity just counts the number
    of common non-zero components of the two vectors and divides this number by N, where N is the length
    of the vectors. If normed=False, the division by N is omitted.
    '''
    # Get the list of shared_items
    si={}
    commons=0
    for item in prefs[person1]: 
        if prefs[person1][item]==1 and prefs[person2][item]==1:   
            commons+=1
    #print commons
    if not normed:
        return commons
    else:
        return commons*1.0/len(prefs[person1]) 

**Aufgabe:**
1. Geben Sie die euklidische Ähnlichkeit und die Pearson Ähnlichkeit zwischen den Personen _Toby_ und _Lisa Rose_ aus.

In [74]:
round_euclid = round(sim_euclid(critics, 'Toby', 'Lisa Rose'), 3)
round_pearson = round(sim_pearson(critics, 'Toby', 'Lisa Rose'), 3)

print("Euklidische Ähnlichkeit: ", round_euclid)
print("Pearson Ähnlichkeit: ", round_pearson)

Euklidische Ähnlichkeit:  0.616
Pearson Ähnlichkeit:  0.991


2. Diskutieren Sie die unterschiedlichen Ähnlichkeitswert.

Da die euklidische Ähnlichkeit den absoluten Abstand misst, sieht man, dass die Bewertungen von Toby und Lisa Rose deutlich voneinander abweichen. Misst man allerdings die Pearson-Ähnlichkeit (welche den linearen Zusammenhang misst), stellt sich heraus, dass es einen beinahen perfekten linearen Zusammenhang gibt. Dies bedeutet, dass Toby und Lisa Rose einen fast gleichen Geschmack haben, die Bewertungsskala in absoluten Werten allerdings anders benutzen und die eine Person systematisch strenger bewertet als die andere.

__Aufgabe:__
0. Schreiben Sie eine Funktion `topMatches(prefs,person,similarity)`, welche für eine beliebige in _critics_ enthaltene Person die Ähnlichkeitswerte zu allen anderen Personen berechnet und in einer geordneten Liste zurück gibt. Der Funktion soll als Übergabeparameter auch die anzuwendende Ähnlichkeitsfunktion (`sim_euclid` oder `sim_pearson`) übergeben werden können. Berechnen Sie mit dieser Funktion für jede Person die *top matches*, zunächst unter Verwendung der euklidischen- dann unter Verwendung der Pearson-Ähnlichkeit.
1. Geben Sie mit der implementierten Funktion die *top matches* der Person Toby aus.

In [75]:
def topMatches(prefs,person,similarity):
    similarities=[]

    for persons in prefs:
        if persons == person:
            continue
        sim = similarity(prefs, person, persons)
        similarities.append((round(sim, 3), persons))

    similarities.sort()
    similarities.reverse()
    return similarities

In [76]:
print("Euklidische Ähnlichkeit: " + str(topMatches(critics,'Toby',sim_euclid)))

Euklidische Ähnlichkeit: [(0.667, 'Mick LaSalle'), (0.625, 'Claudia Puig'), (0.616, 'Lisa Rose'), (0.558, 'Michael Phillips'), (0.523, 'Jack Matthews'), (0.511, 'Gene Seymour')]


In [77]:
print("Pearson Ähnlichkeit: " + str(topMatches(critics,'Toby',sim_pearson)))

Pearson Ähnlichkeit: [(0.991, 'Lisa Rose'), (0.924, 'Mick LaSalle'), (0.893, 'Claudia Puig'), (0.663, 'Jack Matthews'), (0.381, 'Gene Seymour'), (-1.0, 'Michael Phillips')]


2. Vergleichen Sie die beiden Ähnlichkeitsmaße. Welches Ähnlichkeitsmaß erscheint Ihnen für diesen Anwendungsfall sinnvoller und warum?

In diesem Anwendungsfall scheint die Pearson-Ähnlichkeit eindeutig die bessere zu sein. Der wichtigste Grund dafür ist, dass das Benutzerverhalten relativ betrachtet wird. Während ein kritischer Benutzer einem gutem Film die Bewertung von drei Sternen gibt, gibt ein nachsichtiger Benutzer einem guten Film 5 Sterne. Während hier die absoluten Zahlen stärker voneinander abweichen, wird bei Pearson nur auf die relative Abweichung des Mittelwerts geschaut. Zudem kann Pearson auch negative Werte annehmen, somit können auch gegenteilige Geschmäcker bestimmt werden.

### Berechnung von Empfehlungen mit User basiertem Collaborative Filtering
Für die Produkte, die von einer Person noch nicht gekauft wurden, sollen Empfehlungen berechnet werden. Die Empfehlungen können ebenfalls Werte zwischen 1 (wird nicht empfohlen) und 5 (wird stark empfohlen) annehmen. Für die Berechnung der Empfehlung werden die Bewertungen des jeweiligen Produkts durch die anderen Personen herangezogen. Dabei werden die Bewertungen der ähnlichen Personen (d.h. hoher Pearson-Korrelationswert) stärker mit einbezogen als die Bewertungen durch Personen mit einem niedrigen Korrelationswert.

__Beispiel:__
Toby hat die Filme *The Night Listener*, _Lady in the Water_ und _Just My Luck_ noch nicht gekauft. Für diese Filme soll für Toby eine Empfehlung berechnet werden.
In der unten aufgeführten Tabelle enthält die zweite Spalte die _Pearson-Ähnlichkeitswerte_ zwischen Toby und den anderen Personen. Die Spalten 3, 5 und 7 enthalten die Bewertungen der Filme *The Night Listener*, _Lady in the Water_ und _Just My Luck_ durch die anderen Personen. Die Spalten 4, 6 und 8 enthalten die jeweilige Filmbewertung gewichtet (mulipliziert) mit den Ähnlichkeitswerten der jeweiligen Person. Es fällt auf, dass in der Tabelle _Michael_ nicht enthalten ist. Das liegt daran, dass _Michael_ und _Toby_ einen negativen Ähnlichkeitswert aufweisen, d.h. deren Interessen sind gegenläufig. Personen mit negativem Ähnlichkeitswert sollten für Empfehlungen nicht berücksichtigt werden.
Die Zeile _Sum_ enthält die Summe aller gewichteten Bewertungen. Aus diesem Wert allein kann die Empfehlung noch nicht abgeleitet werden, da Filme die nur von wenigen Personen bewertet wurden, eine relativ kleine Summe ergeben. Deshalb sollte _Sum_ noch durch die Anzahl der Bewertungen für diesen Film geteilt werden. Oder besser: Nicht durch die Summe der Bewertungen, sondern durch die Summe der relevanten Ähnlichkeitswerte (*KSum*). Der resultierende Empfehlungswert ist in der letzten Zeile eingetragen.



<img src="https://maucher.home.hdm-stuttgart.de/Pics/recommenderFilmCalculation.PNG" style="width:500px" align="center">




__Aufgabe:__
Schreiben Sie eine Funktion *getRecommendations(prefs,person,similarity)*, mit der die Empfehlungswerte berechnet werden können und bestimmen Sie die Empfehlungswerte für Toby. Der Funktion wird  

* das Dictionary _critics_ mit den Filmbewertungen, 
* der Name der Person, für welche Empfehlungen berechnet werden sollen
* die Methode für die Berechnung der Ähnlichkeit *sim_euclid* oder *sim_pearson*

übergeben. Die Methode soll eine geordnete Liste zurück geben. Jedes Listenelement enthält an erster Stelle den berechneten Empfehlungswert und an zweiter Stelle den Namen des Films. Die Liste soll nach Empfehlungswerten absteigend geordnet sein.

Testen Sie diese Funktion indem Sie die Empfehlungen für _Toby_ berechnen und mit den Werten in der oben aufgeführten Tabelle vergleichen.

In [78]:
def getRecommendations(prefs, person, similarity):
    matches = topMatches(prefs, person, similarity)
    positiv_matches = []

    for i in range(len(matches)):
        if matches[i][0] >= 0:
            positiv_matches.append(matches[i])

    films = []

    for i in range(len(positiv_matches)):
        for item in prefs[positiv_matches[i][1]]:
            if item not in films:
                if item not in prefs[person] or prefs[person][item] == 0:
                    films.append(item)

    recommendations = []
    for film in films:
        sum = 0
        ksum = 0

        for i in range(len(positiv_matches)):
            if film in prefs[positiv_matches[i][1]]:
                sum += positiv_matches[i][0] * prefs[positiv_matches[i][1]][film]
                ksum += positiv_matches[i][0] 
        
        result = sum / ksum

        recommendations.append((round(result, 3), film)) 

    recommendations.sort()
    recommendations.reverse()
    return recommendations

In [79]:
print("Recommendations mit Pearson Ähnlichkeit: " + str(getRecommendations(critics, "Toby", sim_pearson)))

Recommendations mit Pearson Ähnlichkeit: [(3.348, 'The Night Listener'), (2.833, 'Lady in the Water'), (2.531, 'Just My Luck')]


In [80]:
print("Recommendations mit Euklidische Ähnlichkeit: " + str(getRecommendations(critics, "Toby", sim_euclid)))

Recommendations mit Euklidische Ähnlichkeit: [(3.427, 'The Night Listener'), (2.796, 'Lady in the Water'), (2.407, 'Just My Luck')]


### Berechnung von Empfehlungen mit Item basiertem Collaborative Filtering
In den vorigen Aufgaben wurden Ähnlichkeiten zwischen Personen bestimmt und für Produktempfehlungen benutzt (User basiertes Collaborative Filtering). Jetzt soll die Ähnlichkeit zwischen Produkten berechnet werden und auf der Basis dieser Produktähnlichkeit Empfehlungen berechnet werden (Item basiertes Collaborative Filtering).

Dabei sollen die bereits implementierten Ähnlichkeitsfunktion *sim_euclid* und *sim_pearson* sowie die Ähnlichkeeits-Sortierfunktion *topMatches* unverändert eingesetzt werden.

__Aufgabe:__

1. Implementieren Sie eine Funktion, welche das Bewertungsdictionary *critics* derart transformiert, dass die Funktionen `sim_euclid`, `sim_pearson` und `topMatches` für das Item-basierte CF unverändert eingesetzt werden können. Die transformierte Matrix soll unter dem Namen *transCritics* abgespeichert werden.
2. Schreiben Sie eine Funktion `calculateSimilarItems`, die aus der transformierten Matrix *transCritics* ein Dictionary berechnet, welches die Ähnlichkeit zwischen allen Filmen beschreibt. Die Keys des Dictionary sind die Filmnamen. Die Values sind geordnete Listen, welche die Funktion `topMatches` zurückgibt, wenn sie für die Filme (nicht für die User) aufgerufen wird. Dieses Dictionary wird an das aufrufende Programm zurück geben. 
3. Schreiben Sie eine Funktion `getRecommendedItems`, welche basierend auf dem im unten aufgeführten Beispiel dargestellten Verfahren unter Vorgabe der Bewertungsmatrix und der zu verwendenden Ähnlichkeitsfunktion Produktempfehlungen berechnet.
4. Testen Sie die Funktion indem Sie die Empfehlungen für Toby berechnen und mit den Werten in der unten aufgeführten Tabelle vergleichen

__Erläuterndes Beispiel:__

_Toby_ hat die Filme *The Night Listener*, *Lady in the Water* und *Just My Luck* noch nicht gekauft. Für diese Filme soll für *Toby* eine Empfehlung berechnet werden. Gekauft und bewertet hat *Toby* die Filme *Snakes on a plane*, *Superman Returns* und *You and me and Dupree*. Diese bereits vorhandenen Filme bilden die erste Spalte der unten dargestellten Matrix. In der zweiten Spalte befinden sich _Toby's_ Bewertungen dieser Filme. Die Spalten 3,5 und 7 enthalten die Ähnlichkeitswerte (mit *calculateSimilarItems* unter Verwendung des normierten euklidischen Ähnlichkeitsmaßes berechnet) zwischen den drei von *Toby* noch nicht gekauften Filmen und den drei von _Toby_ bewerteten Filmen. Diese Ähnlichkeitswerte werden jeweils mit _Toby's_ Bewertungen multipliziert. Das Resultat dieser Multiplikation befindet sich in den Spalten 4,6 und 8. Der finale Empfehlungswert für die von _Toby_ noch nicht gekauften Filme wird berechnet in dem in den Spalten 4,6 und 8 zunächst die Summe über die Werte dieser Spalte in den drei oberen Zeilen berechnet wird und durch die Summe über die Werte der Spalten 3,5 und 7 geteilt wird. Im Fall, dass die *Pearson-Korrelation* zwischen den Filmen als Ähnlichkeitswert herangezogen wird, können negative Ähnlichkeitswerte auftreten. Dann soll in die Berechnung eines Empfehlungswert für Film A nur dann die Bewertung von Film B einfließen, wenn der Korrelationswert zwischen beiden $>0$ ist.  


<img src="https://maucher.home.hdm-stuttgart.de/Pics/recommenderFilmItemBased.PNG" style="width:500px" align="center">


1. Implementieren Sie eine Funktion, welche das Bewertungsdictionary *critics* derart transformiert, dass die Funktionen `sim_euclid`, `sim_pearson` und `topMatches` für das Item-basierte CF unverändert eingesetzt werden können. Die transformierte Matrix soll unter dem Namen *transCritics* abgespeichert werden.

In [81]:
def transformCritics(prefs):
    transCritics = {}
    for person in prefs:
        for film in prefs[person]:
            if film not in transCritics:
                transCritics[film] = {} 
            transCritics[film][person] = prefs[person][film]
 
    return transCritics

In [82]:
transCritics = transformCritics(critics)
print("Item-basierte CF", transCritics)

Item-basierte CF {'Lady in the Water': {'Lisa Rose': 2.5, 'Gene Seymour': 3.0, 'Michael Phillips': 2.5, 'Mick LaSalle': 3.0, 'Jack Matthews': 3.0}, 'Snakes on a Plane': {'Lisa Rose': 3.5, 'Gene Seymour': 3.5, 'Michael Phillips': 3.0, 'Claudia Puig': 3.5, 'Mick LaSalle': 4.0, 'Jack Matthews': 4.0, 'Toby': 4.5}, 'Just My Luck': {'Lisa Rose': 3.0, 'Gene Seymour': 1.5, 'Claudia Puig': 3.0, 'Mick LaSalle': 2.0}, 'Superman Returns': {'Lisa Rose': 3.5, 'Gene Seymour': 5.0, 'Michael Phillips': 3.5, 'Claudia Puig': 4.0, 'Mick LaSalle': 3.0, 'Jack Matthews': 5.0, 'Toby': 4.0}, 'You, Me and Dupree': {'Lisa Rose': 2.5, 'Gene Seymour': 3.5, 'Claudia Puig': 2.5, 'Mick LaSalle': 2.0, 'Jack Matthews': 3.5, 'Toby': 1.0}, 'The Night Listener': {'Lisa Rose': 3.0, 'Gene Seymour': 3.0, 'Michael Phillips': 4.0, 'Claudia Puig': 4.5, 'Mick LaSalle': 3.0, 'Jack Matthews': 3.0}}


2. Schreiben Sie eine Funktion `calculateSimilarItems`, die aus der transformierten Matrix *transCritics* ein Dictionary berechnet, welches die Ähnlichkeit zwischen allen Filmen beschreibt. Die Keys des Dictionary sind die Filmnamen. Die Values sind geordnete Listen, welche die Funktion `topMatches` zurückgibt, wenn sie für die Filme (nicht für die User) aufgerufen wird. Dieses Dictionary wird an das aufrufende Programm zurück geben. 

In [83]:
def calculateSimilarItems(transCritics, similarity):
    similarities = {}

    for film in transCritics:
        similarities[film] = topMatches(transCritics, film, similarity)

    return similarities

In [84]:
print("Pearson Ähnlichkeit: ", calculateSimilarItems(transCritics, sim_pearson))
print("Euklidische Ähnlichkeit: ", calculateSimilarItems(transCritics, sim_euclid))

Pearson Ähnlichkeit:  {'Lady in the Water': [(0.764, 'Snakes on a Plane'), (0.488, 'Superman Returns'), (0.333, 'You, Me and Dupree'), (-0.612, 'The Night Listener'), (-0.945, 'Just My Luck')], 'Snakes on a Plane': [(0.764, 'Lady in the Water'), (0.112, 'Superman Returns'), (-0.333, 'Just My Luck'), (-0.566, 'The Night Listener'), (-0.645, 'You, Me and Dupree')], 'Just My Luck': [(0.556, 'The Night Listener'), (-0.333, 'Snakes on a Plane'), (-0.423, 'Superman Returns'), (-0.486, 'You, Me and Dupree'), (-0.945, 'Lady in the Water')], 'Superman Returns': [(0.658, 'You, Me and Dupree'), (0.488, 'Lady in the Water'), (0.112, 'Snakes on a Plane'), (-0.18, 'The Night Listener'), (-0.423, 'Just My Luck')], 'You, Me and Dupree': [(0.658, 'Superman Returns'), (0.333, 'Lady in the Water'), (-0.25, 'The Night Listener'), (-0.486, 'Just My Luck'), (-0.645, 'Snakes on a Plane')], 'The Night Listener': [(0.556, 'Just My Luck'), (-0.18, 'Superman Returns'), (-0.25, 'You, Me and Dupree'), (-0.566, 'Sn

3. Schreiben Sie eine Funktion `getRecommendedItems`, welche basierend auf dem im unten aufgeführten Beispiel dargestellten Verfahren unter Vorgabe der Bewertungsmatrix und der zu verwendenden Ähnlichkeitsfunktion Produktempfehlungen berechnet.

In [85]:
def getRecommendedItems(prefs, person, similarity):
    critics = calculateSimilarItems(prefs, similarity)

    personRatings = []
    for movie, rating in prefs.items():
        if person in rating:
            personRatings.append((movie, rating[person]))

    rated_movies = []

    for rated_movie in personRatings:
        rated_movies.append(rated_movie[0])

    recommendations = []
    for movie in critics: 
        if movie in rated_movies:
            continue  

        sum = 0
        ksum = 0

        for rated_movie in personRatings:
            for similar_rating, similar_movie in critics[movie]:
                if similar_movie == rated_movie[0]:
                    if similar_rating > 0:
                        sum += similar_rating * rated_movie[1]
                        ksum += similar_rating

        if ksum > 0:
            recommendations.append((round(sum / ksum, 3), movie))

        recommendations.sort()
        recommendations.reverse()
    return recommendations

4. Testen Sie die Funktion indem Sie die Empfehlungen für Toby berechnen und mit den Werten in der unten aufgeführten Tabelle vergleichen

In [86]:
print("Pearson Ähnlichkeit: ", getRecommendedItems(transCritics, "Toby", sim_pearson))
print("Euklidische Ähnlichkeit: ", getRecommendedItems(transCritics, "Toby", sim_euclid))

Pearson Ähnlichkeit:  [(3.611, 'Lady in the Water')]
Euklidische Ähnlichkeit:  [(3.205, 'The Night Listener'), (3.082, 'Lady in the Water'), (3.042, 'Just My Luck')]


## last.fm Musikempfehlungen

__Aufgabe:__

1. Installieren Sie das Package pylast. Stellen Sie durch Aufruf der Funktion *network=pylast.LastFMNetwork()* eine Verbindung zu *last.fm* her. Beim Aufruf der Funktion wird die Anmeldung und Authentifizierung durchgeführt. Die Funktion gibt ein Objekt der Klasse *Network* zurück. Über dieses Objekt können Methoden, wie

    * *get_artist("kuenstlerName")* (liefert Objekt der Klasse *Artist*)
    * *get_album("albumName")* (liefert Objekt der Klasse *Album*)
    * *get_track("songName")* (liefert Objekt der Klasse *Track*)
    * *get_user("userName"):* (liefert Objekt der Klasse *Tag*)
    * usw.
    
      aufgerufen werden. Die Menge aller verfügbaren Klassen und deren Attribute und Methoden können dem Modul _pylast.py_ entnommen werden. In der untenstehenden Codel Zelle können Sie beispielsweise über das *Network*-Objekt die Methode `get_artist("BandIhrerWahl")` aufrufen um die von lastFM berechneten ähnlichsten Bands anzuzeigen.

2. Führen Sie die untenstehende Code Zelle aus um eine vordefinierte Gruppe von Benutzern, passend zu Band 'Slipknot' zu bekommen. Falls Sie ein lastfm Konto haben können Sie manuell auch andere Bands und Benutzer eintragen.
3. Implementieren Sie eine Funktion `createLastfmUserDict()`. Dieser Funktion soll, die oben angelegte Liste von *User*-Objekten _group_ übergeben werden. Für jeden User in *group* sollen die 20 beliebtesten Bands mit der Methode `topartists=*User*.get_top_artists()[0:20]` bestimmt werden. Die Methode gibt eine Liste von *Artist*-Objekt/Gewichtung-Paaren zurück. Die Gewichtungen von Objekten werden in diesem Versuch nicht benötigt. Auf das *i.te* *Artist*-Objekt selbst kann mit `topartists[i].item` zugegriffen werden. Die Menge aller Bands, die auf diese Weise gesammelt werden, wird im folgenden mit _AllBands_ bezeichnet. D.h. in *AllBands* befinden sich alle Bands, die für mindestens einen User in *group* zu den Top-20 gehören. Nun soll ein verschachteltes Dictionary mit Namen *userDict* wie folgt angelegt werden:

    * Die Keys sind die Namen der _User_-Objekte in _group_. Auf den Namen eines Objekts kann mit `get_name()` zugegriffen werden.
    * Die Values sind selbst wieder Dictionaries, deren Keys die Namen der Bands in *AllBands* sind. Achten Sie auch hier darauf, dass Sie nicht das *Artist*-Objekt selbst, sondern dessen Namen als Key verwenden. 
    * Für den User *a* und die Band *b* ist der Value `userDict[a][b]= 1`, falls *b* zu den Top-20 des Users *a* gehört. Andernfalls ist `userDict[a][b]= 0`. 
    
    Das derart angelegte Dictionary soll von der Funktion zurückgegeben werden. 
4. Wählen Sie jetzt einen beliebigen User aus *group*. Bestimmen Sie zu diesem User die ähnlichsten User in *group* durch Anwendung der im ersten Teilversuch implementierten Funktion `topMatches()`. Der Funktion wird das angelegte *userDict* und der Name des gewählten Users übergeben. Als Ähnlichkeitsmaß soll die euklidische Metrik angewandt werden.
5. Bestimmen Sie dann für den gewählten User Band-Empfehlungen durch Anwendung der im ersten Teilversuch implementierten Funktion `getRecommendations()`. Der Funktion wird das angelegte *userDict* und der Name des gewählten Users übergeben. Als Ähnlichkeitsmaß soll die euklidische Metrik, danach die Russel_Rao Metrik, angewandt werden.     
6. Diskutieren Sie das Ergebnis

In [87]:
import pylast
nw=pylast.LastFMNetwork(api_key = "993a5bd9d79a98a53677570368d55acd",api_secret = "9b8de0b57903ac007cdd8ec9003b341e",username = "pythonlab")
band='Slipknot'
art1 = nw.get_artist(band)
print("Most similar (as calculated by lastFM) for artist: ",band)
for it in art1.get_similar(5):
    print("%3.3f \t %s"%(it.match, it.item))

print("\nApply predefined group of users")
usernames=['BrunoJoS','DPREBOYE','MPistol40','NemoNightfall','SkyRif','Wags1382','Znapsen','cortapsyco','emill_67','sattuviitana']
group=[]
for u in usernames:
    u1 = nw.get_user(u)
    group.append(u1)
print(group)

Most similar (as calculated by lastFM) for artist:  Slipknot
1.000 	 Korn
0.681 	 System of a Down
0.591 	 Mudvayne
0.578 	 Stone Sour
0.556 	 Vended

Apply predefined group of users
[pylast.User('BrunoJoS', pylast.LastFMNetwork('993a5bd9d79a98a53677570368d55acd', '9b8de0b57903ac007cdd8ec9003b341e', '', 'pythonlab', '')), pylast.User('DPREBOYE', pylast.LastFMNetwork('993a5bd9d79a98a53677570368d55acd', '9b8de0b57903ac007cdd8ec9003b341e', '', 'pythonlab', '')), pylast.User('MPistol40', pylast.LastFMNetwork('993a5bd9d79a98a53677570368d55acd', '9b8de0b57903ac007cdd8ec9003b341e', '', 'pythonlab', '')), pylast.User('NemoNightfall', pylast.LastFMNetwork('993a5bd9d79a98a53677570368d55acd', '9b8de0b57903ac007cdd8ec9003b341e', '', 'pythonlab', '')), pylast.User('SkyRif', pylast.LastFMNetwork('993a5bd9d79a98a53677570368d55acd', '9b8de0b57903ac007cdd8ec9003b341e', '', 'pythonlab', '')), pylast.User('Wags1382', pylast.LastFMNetwork('993a5bd9d79a98a53677570368d55acd', '9b8de0b57903ac007cdd8ec9003b34

3. Implementieren Sie eine Funktion `createLastfmUserDict()`. Dieser Funktion soll, die oben angelegte Liste von *User*-Objekten _group_ übergeben werden. Für jeden User in *group* sollen die 20 beliebtesten Bands mit der Methode `topartists=*User*.get_top_artists()[0:20]` bestimmt werden. Die Methode gibt eine Liste von *Artist*-Objekt/Gewichtung-Paaren zurück. Die Gewichtungen von Objekten werden in diesem Versuch nicht benötigt. Auf das *i.te* *Artist*-Objekt selbst kann mit `topartists[i].item` zugegriffen werden. Die Menge aller Bands, die auf diese Weise gesammelt werden, wird im folgenden mit _AllBands_ bezeichnet. D.h. in *AllBands* befinden sich alle Bands, die für mindestens einen User in *group* zu den Top-20 gehören. Nun soll ein verschachteltes Dictionary mit Namen *userDict* wie folgt angelegt werden:

    * Die Keys sind die Namen der _User_-Objekte in _group_. Auf den Namen eines Objekts kann mit `get_name()` zugegriffen werden.
    * Die Values sind selbst wieder Dictionaries, deren Keys die Namen der Bands in *AllBands* sind. Achten Sie auch hier darauf, dass Sie nicht das *Artist*-Objekt selbst, sondern dessen Namen als Key verwenden. 
    * Für den User *a* und die Band *b* ist der Value `userDict[a][b]= 1`, falls *b* zu den Top-20 des Users *a* gehört. Andernfalls ist `userDict[a][b]= 0`. 
    
    Das derart angelegte Dictionary soll von der Funktion zurückgegeben werden. 

In [88]:
def createLastfmUserDict(group):
    userDict = {} 
    user_topartists = {}
    all_bands = []

    for user in group:
        topartists = user.get_top_artists()[0:20] 

        username = user.get_name()
        artist_names = []

        for artist in topartists: 
            artist_name = artist.item.get_name()

            if artist_name not in all_bands: 
                all_bands.append(artist_name)

            artist_names.append(artist_name)
            
        user_topartists[username] = artist_names
        
    for user in group:
        username = user.get_name()
        userDict[username] = {}

        for band in all_bands:
            if band in user_topartists[username]:
                userDict[username][band] = 1
            else:
                userDict[username][band] = 0

    return userDict

In [89]:
userDict = createLastfmUserDict(group)
userDict 

{'BrunoJoS': {'Muse': 1,
  'Linkin Park': 1,
  'Slipknot': 1,
  'Arctic Monkeys': 1,
  'Queens of the Stone Age': 1,
  'Radiohead': 1,
  'Rammstein': 1,
  'Papa Roach': 1,
  'Foo Fighters': 1,
  'Sum 41': 1,
  'Green Day': 1,
  'The Killers': 1,
  'System of a Down': 1,
  'Royal Blood': 1,
  'Madonna': 1,
  '30 Seconds to Mars': 1,
  'Three Days Grace': 1,
  'Pitty': 1,
  'Lady Gaga': 1,
  'M.I.A.': 1,
  'Warpaint': 0,
  'DIR EN GREY': 0,
  'The White Stripes': 0,
  'James Blake': 0,
  'Daughter': 0,
  'Dr. Dre': 0,
  'Beach House': 0,
  'Atoms for Peace': 0,
  'Damien Rice': 0,
  'Die Antwoord': 0,
  'Mogwai': 0,
  'Rage Against the Machine': 0,
  'Hozier': 0,
  'The Band': 0,
  'Nirvana': 0,
  'Wu-Tang Clan': 0,
  'Cat Power': 0,
  'N.W.A': 0,
  'In Flames': 0,
  'Deftones': 0,
  'Korn': 0,
  'Red Hot Chili Peppers': 0,
  'Dark Tranquillity': 0,
  'Soilwork': 0,
  'Limp Bizkit': 0,
  '36 Crazyfists': 0,
  'Soulfly': 0,
  'Stone Sour': 0,
  'Scar Symmetry': 0,
  'Nine Inch Nails': 0,


4. Wählen Sie jetzt einen beliebigen User aus *group*. Bestimmen Sie zu diesem User die ähnlichsten User in *group* durch Anwendung der im ersten Teilversuch implementierten Funktion `topMatches()`. Der Funktion wird das angelegte *userDict* und der Name des gewählten Users übergeben. Als Ähnlichkeitsmaß soll die euklidische Metrik angewandt werden.

In [90]:
bruno = userDict["BrunoJoS"]
print("Top Matches mit Euklidische Ähnlichkeit: ", topMatches(userDict, "BrunoJoS", sim_euclid)) 

Top Matches mit Euklidische Ähnlichkeit:  [(0.966, 'NemoNightfall'), (0.964, 'SkyRif'), (0.963, 'sattuviitana'), (0.963, 'emill_67'), (0.962, 'cortapsyco'), (0.962, 'Znapsen'), (0.962, 'MPistol40'), (0.962, 'DPREBOYE'), (0.961, 'Wags1382')]


5. Bestimmen Sie dann für den gewählten User Band-Empfehlungen durch Anwendung der im ersten Teilversuch implementierten Funktion `getRecommendations()`. Der Funktion wird das angelegte *userDict* und der Name des gewählten Users übergeben. Als Ähnlichkeitsmaß soll die euklidische Metrik, danach die Russel_Rao Metrik, angewandt werden.  

In [91]:
print("Recommendations mit Euklidischer Ähnlichkeit: ", getRecommendations(userDict, "BrunoJoS", sim_euclid))

Recommendations mit Euklidischer Ähnlichkeit:  [(0.556, 'Korn'), (0.445, 'Five Finger Death Punch'), (0.444, 'Rage Against the Machine'), (0.334, 'Stone Sour'), (0.333, 'Red Hot Chili Peppers'), (0.333, 'Metallica'), (0.223, 'Trivium'), (0.222, 'Sabaton'), (0.222, 'Pink Floyd'), (0.222, 'Pendulum'), (0.222, 'Limp Bizkit'), (0.222, 'Lamb of God'), (0.222, 'In Flames'), (0.222, 'Coheed and Cambria'), (0.222, 'Avenged Sevenfold'), (0.111, 'deadmau5'), (0.111, 'blink-182'), (0.111, 'Wu-Tang Clan'), (0.111, 'Wintersun'), (0.111, 'White Zombie'), (0.111, 'Ween'), (0.111, 'Warpaint'), (0.111, 'Volbeat'), (0.111, 'Turmion Kätilöt'), (0.111, 'Tool'), (0.111, 'Theory of a Deadman'), (0.111, 'The Wonder Years'), (0.111, 'The White Stripes'), (0.111, 'The Dear Hunter'), (0.111, 'The Band'), (0.111, 'Tenacious D'), (0.111, 'Team Sleep'), (0.111, 'Stam1na'), (0.111, 'Soulfly'), (0.111, 'Sonata Arctica'), (0.111, 'Soilwork'), (0.111, 'Sick Puppies'), (0.111, 'Shinedown'), (0.111, 'Seether'), (0.111, 

In [92]:
print("Recommendations mit Russel Rao Ähnlichkeit: ", getRecommendations(userDict, "BrunoJoS", sim_RusselRao))

Recommendations mit Russel Rao Ähnlichkeit:  [(0.56, 'Korn'), (0.56, 'Five Finger Death Punch'), (0.482, 'Stone Sour'), (0.44, 'Rage Against the Machine'), (0.361, 'Trivium'), (0.361, 'Red Hot Chili Peppers'), (0.361, 'Metallica'), (0.283, 'Sabaton'), (0.241, 'Sick Puppies'), (0.241, 'Oh, Sleeper'), (0.241, 'Mumford & Sons'), (0.241, 'Memphis May Fire'), (0.241, 'Lamb of God'), (0.241, 'Bullet for My Valentine'), (0.241, 'Bring Me the Horizon'), (0.241, 'Breaking Benjamin'), (0.241, 'Billy Talent'), (0.241, 'Ben Howard'), (0.241, 'All Time Low'), (0.241, 'Alexisonfire'), (0.199, 'Pink Floyd'), (0.199, 'In Flames'), (0.199, 'Avenged Sevenfold'), (0.163, 'Wintersun'), (0.163, 'Tenacious D'), (0.163, 'Powerwolf'), (0.163, 'Pentakill'), (0.163, 'Howard Shore'), (0.163, 'Hans Zimmer'), (0.163, 'DragonForce'), (0.163, 'Arch Enemy'), (0.163, 'Amon Amarth'), (0.157, 'Pendulum'), (0.157, 'Limp Bizkit'), (0.12, 'deadmau5'), (0.12, 'Ween'), (0.12, 'Volbeat'), (0.12, 'Turmion Kätilöt'), (0.12, 'Th

6. Diskutieren Sie das Ergebnis